In [ ]:
%autosave 0 

# Vertical Sesimic Profiling inversion

We are going to focus our attention to a common approach when dealing with geophysical inverse problems, regularization and prior information. In this example we will invert synthetic data from a vertical seismic profiling survey. To do so, we will discretize the following equation:
\begin{align}
t(z) =  \int_{0}^{z} \frac{1}{v(z')}dz',
\end{align}
where $t(z)$ is the traveltime from the surface to the depth $z$ and $v(z')$ represents the propagation speed of the medium. To make the problem linear we will parameterize the problem using slowness instead of speed. The discrete version for a regular $z$ sampling of it can be written as follows:
\begin{align}
t_i =  \sum_{j=0}^{N_i} s_j \Delta z,
\end{align}
where $\Delta z$ represents the sampling interval in the $z$ direction, while $t_i$ and $s_j$ are the traveltime and slowness at $z_i=N_i \Delta z$ depth, respectively.
In this example we will assume that the true subsurface vertical speed is given by the following equation:
\begin{align}
v(z)=3000 + \sqrt{1000 z},
\end{align}
expressed in km/s.

In [ ]:
#Adding library modules to PYTHONPATH
import sys
sys.path.append("../python")
import numpy as np
#Plotting library
from matplotlib import pyplot as plt
import matplotlib
from mpl_toolkits.axes_grid1 import make_axes_locatable
%matplotlib inline
params = {
    'image.cmap': 'gray',
    'axes.grid': False,
    'savefig.dpi': 300,  # to adjust notebook inline plot size
    'axes.labelsize': 14, # fontsize for x and y labels (was 10)
    'axes.titlesize': 14,
    'font.size': 14, 
    'legend.fontsize': 12,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14
}
matplotlib.rcParams.update(params)
#Inversion library-related modules
import pyVector as Vec
import pyOperator as Op
from pyLinearSolver import LCGsolver as LCG
import pyProblem as Prblm
from pyStopper import BasicStopper as Stopper

In [ ]:
#Definition of the modeling operator forward and its adjoint
class VSP_op(Op.Operator):
    """
       Vertical Sesimic Profiling operator      
    """
    
    def __init__(self,model,data,dz,desampling):
        """Operator constructor"""
        self.setDomainRange(model,data)
        self.dz = dz                           #Sampling in depth
        self.desampling = int(desampling)      #Desampling of the data points (should be 1 or greater)
        self.M = model.getNdArray().shape[0]   #Number of model points
        self.N = data.getNdArray().shape[0]    #Number of data points
        if((self.M-2) < (self.N-2)*self.desampling): 
            raise ValueError("ERROR! Too many data points! Change desampling or number of data points")
        return
    
    def forward(self,add,model,data):
        """
           Modeling operator from slowness to traveltime
           add     = [no default] - boolean; Flag to add modeled data to input vector
           model   = [no default] - vector class; slowness model vector
           data    = [no default] - vector class; traveltime data vector
        """
        self.checkDomainRange(model,data)
        if(not add): data.zero()     #data = 0
        modelNd = model.getNdArray() #Getting pointer to Numpy model array
        dataNd = data.getNdArray()   #Getting pointer to Numpy data array
        #First data point
        dataNd[0] += modelNd[0]*dz
        for idata in range(1,self.N-1):
            dataNd[idata] += np.sum(modelNd[:(idata)*self.desampling+1])*dz
        #Last data point
        dataNd[-1] += np.sum(modelNd[:])*dz
        return
    
    def adjoint(self,add,model,data):
        """
           Adjoint operator from traveltime to slowness
           add     = [no default] - boolean; Flag to add modeled data to input vector
           model   = [no default] - vector class; slowness model vector
           data    = [no default] - vector class; traveltime data vector
        """
        self.checkDomainRange(model,data)
        if(not add): model.zero()    #model = 0
        modelNd = model.getNdArray() #Getting pointer to Numpy model array
        dataNd = data.getNdArray()   #Getting pointer to Numpy data array
        #First data point
        modelNd[0] += dataNd[0]*dz
        for idata in range(1,self.N-1):
            modelNd[:(idata)*self.desampling+1] += dataNd[idata]*dz
        #Last data point
        modelNd[:] += dataNd[-1]*dz
        return

In [ ]:
dz = 2.
zmax = 1000.0
z = np.linspace(0.0,zmax,int(zmax/dz)+1)
vel = 3000.0 + np.sqrt(1000.0*z)
slowness = 1.0/vel
model_true = Vec.vectorIC(slowness)
#Desampling of receivers
desampling = 1 #20 m sampling
ndata = int(zmax/(dz*desampling))+1
data_true = Vec.vectorIC((ndata,))
VSP_10 = VSP_op(model_true,data_true,dz,desampling)

In [ ]:
VSP_10.forward(False,model_true,data_true)

In [ ]:
VSP_10.dotTest(True)

In [ ]:
plt.plot(data_true.getNdArray(),'*')

In [ ]:
#Create stopping criteria and related object
niter = 2000
Stop  = Stopper(niter=niter)
#Create LCG solver
LCGsolver = LCG(Stop)
LCGsolver.setDefaults(save_obj=True) #Saving objective function within the solver

In [ ]:
#Inital slowness model
model = model_true.clone()
model.zero() #m = 0
VSP_prob = Prblm.ProblemL2Linear(model,data_true,VSP_10)

In [ ]:
LCGsolver.run(VSP_prob,verbose=True)

In [ ]:
plt.plot(VSP_prob.model.getNdArray())